# 🎚️ Chapter 4 — Model selection (hands-on)

Companion to Chapter 4. You'll sweep the complexity dial yourself, watch training error fall
forever while held-out error turns back up, and pick the bottom of the U.

**Learn** (visible checks) → **Apply** (hidden auto-graded, hints) → **Appendix** (locked solutions).


In [ ]:
# Setup — run me first
try:
    import numpy  # noqa
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy'])
import base64, marshal, numpy as np
def check(name, fn, hint=''):
    try: ok=bool(fn())
    except Exception as e: ok=False; hint=(hint+f'  (raised {type(e).__name__}: {e})').strip()
    print(f"{'✅' if ok else '❌'}  {name}" + (f'  — {hint}' if not ok and hint else '')); return ok
def _eq(a,b,tol=1e-6):
    if isinstance(b,(list,tuple)): return isinstance(a,(list,tuple)) and len(a)==len(b) and all(_eq(x,y,tol) for x,y in zip(a,b))
    if isinstance(b,bool): return a is b or a==b
    if isinstance(b,float):
        try: return abs(a-b)<=tol
        except Exception: return False
    return a==b
_TESTS={"p1": "WwMAAAApAqkCWwMAAADpAQAAAOkCAAAA6QMAAABbAwAAAHIBAAAAcgIAAAByAwAAAGcAAAAAAAAAACkCqQJbAgAAAOkAAAAAcgUAAABbAgAAAHIBAAAA6f////9nAAAAAAAA8D8pAqkCWwMAAAByAgAAAOkEAAAA6QYAAABbAwAAAHIBAAAAcggAAADpCQAAAGe/3EgYPTb9Pw==", "p2": "WwMAAAApAqkBe+kBAAAA5wAAAAAAgDJA6QMAAADnAAAAAAAAGkDpBQAAAOfNzMzMzMwSQOkJAAAA5zMzMzMzMxdAMHIFAAAAKQKpAXvpAgAAAOcAAAAAAADwP+kEAAAA5wAAAAAAAOA/MHIMAAAAKQKpAXvpBwAAAOdmZmZmZmYKQDByDwAAAA==", "p3": "WwMAAAApAqkC6QYAAADpAwAAAFsDAAAAKQJbBAAAAOkCAAAAcgIAAADpBAAAAOkFAAAAWwIAAADpAAAAAOkBAAAAKQJbBAAAAHIGAAAAcgcAAAByBAAAAHIFAAAAWwIAAAByAwAAAHICAAAAKQJbBAAAAHIGAAAAcgcAAAByAwAAAHICAAAAWwIAAAByBAAAAHIFAAAAKQKpAukHAAAAcgIAAABbAwAAACkCWwQAAAByAgAAAHIEAAAAcgUAAAByAQAAAFsDAAAAcgYAAAByBwAAAHIDAAAAKQJbBQAAAHIGAAAAcgcAAAByAwAAAHIFAAAAcgEAAABbAgAAAHICAAAAcgQAAAApAlsFAAAAcgYAAAByBwAAAHIDAAAAcgIAAAByBAAAAFsCAAAAcgUAAAByAQAAACkCqQJyBAAAAHIDAAAAWwIAAAApAlsCAAAAcgMAAAByAgAAAFsCAAAAcgYAAAByBwAAACkCWwIAAAByBgAAAHIHAAAAWwIAAAByAwAAAHICAAAA"}
_HINTS={"p1": ["Square each difference, average them, then take the square root.", "Use zip(y_true, y_pred) to pair them up."], "p2": ["scores maps degree -> held-out error; you want the key with the SMALLEST value.", "min(scores, key=lambda k: scores[k])."], "p3": ["Fold sizes: n//k each, and the first n%k folds get one extra so they sum to n.", "Test fold = a contiguous block; train = every other index, in order.", "Return a list of (train_list, test_list) tuples, one per fold."]}
_SOLUTIONS={}
def grade(pid, fn):
    cs=marshal.loads(base64.b64decode(_TESTS[pid]))
    for k,(args,exp) in enumerate(cs,1):
        try: got=fn(*args)
        except Exception as e:
            print(f"❌ {pid}: hidden case {k}/{len(cs)} raised {type(e).__name__}: {e}. Try hint('{pid}')."); return False
        if not _eq(got,exp):
            print(f"❌ {pid}: hidden case {k}/{len(cs)} failed — not right yet. Try hint('{pid}')."); return False
    print(f'✅ {pid}: all {len(cs)} hidden tests passed 🎉'); return True
def hint(pid,n=None):
    hs=_HINTS.get(pid,[]); rng=range(len(hs)) if n is None else [n-1]
    [print(f'Hint {i+1}: {hs[i]}') for i in rng]
def reveal(pid):
    if pid not in _SOLUTIONS: print('🔒 Locked — run the Appendix cell at the bottom first.'); return
    print(base64.b64decode(_SOLUTIONS[pid]).decode())
print('Toolkit ready ✓')


## Part A — Learn: sweep the complexity dial
The same experiment as the chapter's diagram, in code. Fit polynomials of growing degree to a
curved pattern and record **training** and **held-out** error for each.

In [ ]:
rng = np.random.default_rng(0)
truth = lambda x: 30 + 38*np.sin(x/2.6)          # the real pattern (unknown to the model)
Xtr = np.linspace(1.2, 17.5, 14); ytr = truth(Xtr) + rng.normal(0, 4.0, 14)
Xho = np.linspace(2.4, 16.6, 9) + 0.35; yho = truth(Xho) + rng.normal(0, 4.0, 9)

def errors(degree):
    p = np.polynomial.Polynomial.fit(Xtr, ytr, degree)     # rescales x internally
    tr = float(np.sqrt(np.mean((p(Xtr) - ytr)**2)))
    ho = float(np.sqrt(np.mean((p(Xho) - yho)**2)))
    return tr, ho

scores = {}
print(f"{'degree':>7} {'training':>10} {'held-out':>10}")
for d in (1, 3, 5, 9, 14):
    tr, ho = errors(d); scores[d] = ho
    print(f'{d:>7} {tr:>10.2f} {ho:>10.2f}')

best = min(scores, key=lambda d: scores[d])
print(f'\nlowest held-out error at degree {best}')

check('training error falls as degree grows (so it can never pick the model)',
      lambda: errors(14)[0] < errors(1)[0])
check('held-out error is U-shaped: the best degree beats both extremes',
      lambda: scores[best] < scores[1] and scores[best] < scores[14])
check('degree 14 memorises: near-zero training error but far worse held-out',
      lambda: errors(14)[0] < 0.5 and errors(14)[1] > scores[best])


### The takeaway you just measured
Training error went **down all the way** to essentially zero, and the degree-14 model is the worst
one you have. Held-out error is the only number that turned back up — which is why it, and never
training error, decides which model you ship.

## Part B — Apply  🔒
Implement each, run `grade('pN', fn)` (hidden tests). `hint('pN')` if stuck.

### p1 · `rmse(y_true, y_pred)`
Root-mean-squared error: square the differences, average, square-root. Return a float.

In [ ]:
def rmse(y_true, y_pred):
    # TODO
    pass

grade('p1', rmse)
# hint('p1')


### p2 · `best_by_holdout(scores)`
Given `{degree: held_out_error}`, return the degree with the **lowest** held-out error. (The whole chapter in one function.)

In [ ]:
def best_by_holdout(scores):
    # TODO: key with the smallest value
    pass

grade('p2', best_by_holdout)
# hint('p2')


### p3 · `kfold_indices(n, k)`
Return the `k` folds for cross-validation as a list of `(train_indices, test_indices)`. Test folds are contiguous blocks; when `n` isn't divisible by `k`, the **first `n % k` folds get one extra** so every index is used exactly once as test.

In [ ]:
def kfold_indices(n, k):
    # TODO: k contiguous test folds; first n%k folds get one extra
    pass

grade('p3', kfold_indices)
# hint('p3')


### ✅ Score

In [ ]:
import io, contextlib
_pairs=[('p1', rmse), ('p2', best_by_holdout), ('p3', kfold_indices)]
done=0
for pid, fn in _pairs:
    buf=io.StringIO()
    with contextlib.redirect_stdout(buf): done+=grade(pid, fn)
print(f'\nApply score: {done} / 3' + ('  — model selection mastered 🏆' if done==3 else ''))


---
## Appendix — Solutions (locked 🔒)
Try first, then run this and call `reveal('p1')` etc.

In [ ]:
_SOLUTIONS={"p1": "ZGVmIHJtc2UoeV90cnVlLCB5X3ByZWQpOgogICAgcmV0dXJuIChzdW0oKGEtYikqKjIgZm9yIGEsYiBpbiB6aXAoeV90cnVlLHlfcHJlZCkpL2xlbih5X3RydWUpKSoqMC41", "p2": "ZGVmIGJlc3RfYnlfaG9sZG91dChzY29yZXMpOgogICAgIyB0aGUgV0hPTEUgcG9pbnQ6IGNob29zZSBieSBoZWxkLW91dCBlcnJvciwgbmV2ZXIgdHJhaW5pbmcgZXJyb3IKICAgIHJldHVybiBtaW4oc2NvcmVzLCBrZXk9bGFtYmRhIGs6IHNjb3Jlc1trXSk=", "p3": "ZGVmIGtmb2xkX2luZGljZXMobiwgayk6CiAgICBzaXplcz1bbi8vayArICgxIGlmIGkgPCBuJWsgZWxzZSAwKSBmb3IgaSBpbiByYW5nZShrKV0gICAjIGZpcnN0IG4layBmb2xkcyBnZXQgb25lIGV4dHJhCiAgICBvdXQ9W107IHN0YXJ0PTAKICAgIGZvciBzIGluIHNpemVzOgogICAgICAgIHRlc3Q9bGlzdChyYW5nZShzdGFydCwgc3RhcnQrcykpOyB0cz1zZXQodGVzdCkKICAgICAgICBvdXQuYXBwZW5kKChbaSBmb3IgaSBpbiByYW5nZShuKSBpZiBpIG5vdCBpbiB0c10sIHRlc3QpKQogICAgICAgIHN0YXJ0Kz1zCiAgICByZXR1cm4gb3V0"}
print('🔓 Unlocked. reveal("p1") / reveal("p2") / reveal("p3").')
